In [26]:
# Step 1: Import Libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from tensorflow.keras.preprocessing import image

# Step 2: Load and Preprocess Data
csv_path = '../NeoJaundice/chd_jaundice.csv'
df = pd.read_csv(csv_path)

# Extract metadata and target variable
metadata = df[['patient_id', 'gender', 'gestational_age', 'age(day)', 'weight']]
target = df['blood(mg/dL)']

# One-hot encode the 'gender' column
encoder = OneHotEncoder(sparse_output=False, drop='first')
gender_encoded = encoder.fit_transform(metadata[['gender']])

# Separate numeric columns for imputation
numeric_columns = ['gestational_age', 'age(day)', 'weight']
numeric_metadata = metadata[numeric_columns]

# Impute missing values in numeric columns
imputer = SimpleImputer(strategy='mean')
numeric_metadata_imputed = imputer.fit_transform(numeric_metadata)

# Combine imputed numeric metadata with one-hot encoded 'gender'
metadata_encoded = np.hstack((numeric_metadata_imputed, gender_encoded))

# Load and preprocess images
def load_and_preprocess_images(image_folder, image_names, target_size=(224, 224)):
    images = []
    for img_name in image_names:
        img_path = os.path.join(image_folder, img_name)
        img = image.load_img(img_path, target_size=target_size)
        img = image.img_to_array(img) / 255.0  # Normalize to [0, 1]
        images.append(img)
    return np.array(images)

image_folder = "../NeoJaundice/images"
image_names = df['image_idx']
images = load_and_preprocess_images(image_folder, image_names)

# Step 3: Define CNN Architectures with Regularization Techniques

# 1. CNN with Dropout
def build_cnn_with_dropout(input_shape):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),  # Dropout after the first convolutional block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),  # Dropout after the second convolutional block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),  # Dropout before the output layer
        layers.Dense(1)  # Output layer for regression
    ])
    return model

# 2. CNN with L2 Regularization
def build_cnn_with_l2(input_shape):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape,
                      kernel_regularizer=regularizers.l2(0.001)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu',
                      kernel_regularizer=regularizers.l2(0.001)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu',
                      kernel_regularizer=regularizers.l2(0.001)),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.Dense(1)  # Output layer for regression
    ])
    return model

# 3. CNN with Batch Normalization
def build_cnn_with_batchnorm(input_shape):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(1)  # Output layer for regression
    ])
    return model

# Step 4: Extract Features Using CNN

def extract_features(model, images):
    # Extract feature vectors
    feature_vectors = model.predict(images)
    return feature_vectors

# Input shape for CNN
input_shape = images[0].shape

# Build and compile the CNN models
cnn_with_dropout = build_cnn_with_dropout(input_shape)
cnn_with_l2 = build_cnn_with_l2(input_shape)
cnn_with_batchnorm = build_cnn_with_batchnorm(input_shape)

# Extract features using each CNN model
dropout_features = extract_features(cnn_with_dropout, images)
l2_features = extract_features(cnn_with_l2, images)
batchnorm_features = extract_features(cnn_with_batchnorm, images)

# Step 5: Combine Features with Metadata and Train Regression Models

def train_and_evaluate(features, technique_name, target):
    # Combine features with metadata
    combined_features = np.hstack((features, metadata_encoded))

    # Handle missing values in the target variable
    target = target.dropna()
    combined_features = combined_features[~np.isnan(combined_features).any(axis=1)]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(combined_features, target, test_size=0.2, random_state=42)

    # Train a simple regression model (e.g., Linear Regression)
    from sklearn.ensemble import RandomForestRegressor
    regressor = RandomForestRegressor(random_state=42)
    regressor.fit(X_train, y_train)

    # Evaluate the model
    y_pred = regressor.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"{technique_name}:")
    print(f"  - MSE: {mse:.4f}")
    print(f"  - R²: {r2:.4f}")
    print()

# Train and evaluate each model
print("Evaluating Regularization Techniques:")
train_and_evaluate(dropout_features, "Dropout", target)
train_and_evaluate(l2_features, "L2 Regularization", target)
train_and_evaluate(batchnorm_features, "Batch Normalization", target)

C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


70/70 ━━━━━━━━━━━━━━━━━━━━ 8s 116ms/step
70/70 ━━━━━━━━━━━━━━━━━━━━ 8s 112ms/step
70/70 ━━━━━━━━━━━━━━━━━━━━ 13s 179ms/step
Evaluating Regularization Techniques:
Dropout:
  - MSE: 5.6964
  - R²: 0.8060

L2 Regularization:
  - MSE: 5.3202
  - R²: 0.8188

Batch Normalization:
  - MSE: 5.8044
  - R²: 0.8023

